In [ ]:
import numpy as np
import math
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import shapely
import datetime
import colormaps as cmaps
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import cm
from pyproj import Proj, transform

ox.settings.log_console = True

from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Helvetica"
})
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['figure.dpi'] = 600
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'STIXGeneral'

plt.rcParams.update({'font.size': 14})

bgCol = "#212121ff"
bgdy = '#940B13'
lred = '#D10D18'

In [ ]:
### DEFINE CONSTANTS ###
tol = 58.6
no2limAQ = 200 # in ug/m3, as per The Air Quality Standards Regulations 2010
no2traffic = 0.68 # proportion of roadside NO2 caused by motor vehicle traffic
ppb_to_ugm3 = 1.9387 # multiply NO2 in ppb to get ug/m3
m1 = 50 # for plotting: small marker size
m2 = 60 # for plotting: large marker size

mainDir = '' # INSERT WORKING DIRECTORY
dataDir = mainDir + 'inputs/'
imgOutDir = mainDir + 'plots/'

# shape data
lsoa_path = dataDir + 'LSOA_2011_EW_BGC.zip'  # TOO LARGE TO UPLOAD TO GITHUB - DOWNLOAD SHAPE FILES FROM OPEN GEOGRAPHY PORTAL (OPEN ACCESS)
msoa_path = dataDir + 'MSOA_2022_EW_BGC.zip'
sheff_path = dataDir + 'City_Boundary.shp'

# lookup
lsoa_msoa_lookup_path = dataDir + 'LSOA_to_MSOA_2021.csv' # TOO LARGE TO UPLOAD TO GITHUB - DOWNLOAD LOOKUP TABLE FROM OPEN GEOGRAPHY PORTAL (OPEN ACCESS)

# NO2 in 2023 from DEFRA (ug/m3)
defra_mod_path = dataDir + 'DEFRA-sheffield-no2-2023-model.csv'
defra_hourly_path = dataDir + 'DEFRA_AirQualityDataHourly_Tidy.csv'

# SCC sensor data
scc_path = dataDir + 'SCC_no2.csv'

# SCC sensor data (flow)
flow_path = dataDir + 'Flow_Sensors_SUFO_2019.csv'

# NAME OF OUTPUT FILE OF THIS SCRIPT
val_path = dataDir + 'validation_data_processed.csv'

In [ ]:
### LOAD DATA ###
# LSOA to MSOA lookup table
lsoa_msoa_lookup = pd.read_csv(lsoa_msoa_lookup_path)

# NO2 in 2023 from DEFRA (ug/m3)
defra_mod = pd.read_csv(defra_mod_path)
defra_hourly = pd.read_csv(defra_hourly_path)

# SCC sensor data (NO2)
scc = pd.read_csv(scc_path)

# SCC sensor data (flow)
flow = pd.read_csv(flow_path)

# spatial data
lsoas = gpd.read_file(lsoa_path).to_crs('WGS84')
msoas = gpd.read_file(msoa_path).to_crs('WGS84')

sheff = gpd.read_file(sheff_path)
sheff = sheff.to_crs('WGS84')

In [ ]:
def ENtoLL(easting,northing):
    lat,lon = transform(Proj('epsg:27700'), Proj('epsg:4326'), easting, northing)
    return lon,lat

In [ ]:
cmap = mpl.cm.viridis
cmap_reversed = mpl.colormaps['viridis_r']
greys = mpl.cm.Greys
greens = mpl.cm.Greens
oranges = mpl.cm.Oranges
blues = mpl.cm.Blues
purples = mpl.cm.Purples
clwm = mpl.cm.coolwarm
brbg = mpl.cm.BrBG_r

In [ ]:
lsoas = lsoas[lsoas.LSOA11NM.str.contains('Sheffield')].reset_index(drop=True)

In [ ]:
# MODELLED DEFRA NO2 #

In [ ]:
lon,lat = ENtoLL(defra_mod['x'], defra_mod['y'])
defra_mod = defra_mod.assign(lon=lon, lat=lat)

defra_mod['geometry'] = [shapely.Point(xy) for xy in zip(defra_mod.lon, defra_mod.lat)]
defra_mod = gpd.GeoDataFrame(defra_mod, crs="WGS84", geometry=defra_mod['geometry'])

In [ ]:
xy = sheff.get_coordinates()

In [ ]:
# associate LSOAs with difftubes
for index, row in defra_mod.iterrows():
    geom = shapely.Point(row['lon'], row['lat'])
    for lInd, lsoa in lsoas.iterrows():
        polygon = lsoa.geometry
        if polygon.contains(geom):
            defra_mod.loc[index, 'LSOA'] = lsoa['LSOA11CD']
            break

In [ ]:
groups = defra_mod[['LSOA','Total_NO2_23']].groupby('LSOA').mean('Total_NO2_23')

In [ ]:
lsoas = lsoas.merge(groups, how='left', left_on='LSOA11CD', right_on='LSOA')

In [ ]:
# HOURLY DEFRA NO2 #

In [ ]:
defra_hourly['Datetime'] = pd.to_datetime(defra_hourly['Date'], format='%Y-%m-%d', errors='coerce')

In [ ]:
# filter to Feb-Oct 2023 (all days inclusive)
defra_hourly = defra_hourly[(defra_hourly['Datetime'] >= '2023-02-01') & (defra_hourly['Datetime'] <= '2023-10-31')]
# filter to peak hours (7am-9am, 4pm-7pm)
defra_hourly = defra_hourly.loc[defra_hourly['Time'].isin(['07:00:00','08:00:00','09:00:00','16:00:00','17:00:00','18:00:00','19:00:00'])].reset_index(drop=True)

In [ ]:
# infill missing values with nan
defra_hourly.loc[defra_hourly['NO2_BR']=='No data','NO2_BR'] = np.nan
defra_hourly.loc[defra_hourly['NO2_DG']=='No data','NO2_DG'] = np.nan
defra_hourly.loc[defra_hourly['NO2_T']=='No data','NO2_T'] = np.nan

# convert strings to floats
defra_hourly['NO2_BR'] = defra_hourly['NO2_BR'].astype('float')
defra_hourly['NO2_DG'] = defra_hourly['NO2_DG'].astype('float')
defra_hourly['NO2_T'] = defra_hourly['NO2_T'].astype('float')

In [ ]:
pnt_BR = shapely.Point(defra_hourly.loc[0,'Lon_BR'],defra_hourly.loc[0,'Lat_BR'])
pnt_DG = shapely.Point(defra_hourly.loc[0,'Lon_DG'],defra_hourly.loc[0,'Lat_DG'])
pnt_T = shapely.Point(defra_hourly.loc[0,'Lon_T'],defra_hourly.loc[0,'Lat_T'])

# geometry column for averaged data
geom = gpd.GeoSeries([pnt_BR,pnt_DG,pnt_T])

# NO2 column for averaged data
NO2 = [np.mean(defra_hourly['NO2_BR']),np.mean(defra_hourly['NO2_DG']),np.mean(defra_hourly['NO2_T'])]

In [ ]:
hourly_means = gpd.GeoDataFrame(geometry=geom).assign(NO2=NO2)

In [ ]:
# HOURLY SCC NO2 #

In [ ]:
date = []
hour = []
for ii in range(len(scc)):
    val = datetime.datetime.strptime(scc.loc[ii,'DateTime'], "%Y-%m-%d  %H:%M:%S")
    date.append(val)
    hour.append(val.hour)

In [ ]:
scc = scc.assign(DateTime=date, Time=hour)

In [ ]:
# filter to Feb-Oct 2023 (all days inclusive)
scc = scc[(scc['DateTime'] >= '2023-02-01') & (scc['DateTime'] <= '2023-10-31')]
# filter to peak hours (7am-9am, 4pm-7pm)
scc = scc.loc[scc['Time'].isin([7,8,9,16,17,18,19])].reset_index(drop=True)

In [ ]:
# convert strings to floats
scc['Firvale_ppb'] = scc['Firvale_ppb'].astype('float')
scc['KingE_ppb'] = scc['KingE_ppb'].astype('float')
scc['Wicker_ppb'] = scc['Wicker_ppb'].astype('float')
scc['Lowfield_ppb'] = scc['Lowfield_ppb'].astype('float')

In [ ]:
%%capture

lon_F, lat_F = ENtoLL(436990, 390218)
lon_KE, lat_KE = ENtoLL(430977, 380760)
lon_W, lat_W = ENtoLL(435959, 388021)
lon_L, lat_L = ENtoLL(435181, 385366)

pnt_F = shapely.Point(lon_F, lat_F)
pnt_KE = shapely.Point(lon_KE, lat_KE)
pnt_W = shapely.Point(lon_W, lat_W)
pnt_L = shapely.Point(lon_L, lat_L)

In [ ]:
# geometry column for averaged data
geom = gpd.GeoSeries([pnt_F,pnt_KE,pnt_W,pnt_L])

# NO2 column for averaged data
NO2 = [np.mean(scc['Firvale_ppb']),np.mean(scc['KingE_ppb']),np.mean(scc['Wicker_ppb']),np.mean(scc['Lowfield_ppb'])]

In [ ]:
scc_means = gpd.GeoDataFrame(geometry=geom).assign(NO2=NO2)

In [ ]:
scc_means = scc_means.assign(NO2_ugm3 = scc_means['NO2']*ppb_to_ugm3)

In [ ]:
# SCC FLOW DATA #

In [ ]:
flow['geometry'] = [shapely.Point(xy) for xy in zip(flow.Lon, flow.Lat)]
flow = gpd.GeoDataFrame(flow, crs="WGS84", geometry=flow['geometry'])

In [ ]:
# ADD LABELS #

In [ ]:
hourly_means = hourly_means.assign(label=1)
scc_means = scc_means.assign(label=2)

In [ ]:
# ROAD NETWORK #

In [ ]:
# Read & simplify South Yorkshire road network
cf = '["highway"~"motorway|trunk|trunk_link|primary|primary_link|secondary|secondary_link|tertiary|tertiary_link|unclassified"]' # CAR NETWORK FOR PAPER
G = ox.graph_from_place('Sheffield', custom_filter=cf, simplify=True)
remove = [node for node, degree in dict(G.degree()).items() if degree < 2]
G.remove_nodes_from(remove)

In [ ]:
edges = ox.graph_to_gdfs(G, nodes=False)
thicknesses = [None] * len(edges)
for ii in range(len(edges)):
    if edges['highway'].iloc[ii] == 'motorway':
        thicknesses[ii] = 1.25
    if edges['highway'].iloc[ii] == 'motorway_link':
        thicknesses[ii] = 1.25
    if edges['highway'].iloc[ii] == 'primary':
        thicknesses[ii] = 1
    if edges['highway'].iloc[ii] == 'primary_link':
        thicknesses[ii] = 1
    if edges['highway'].iloc[ii] == 'secondary':
        thicknesses[ii] = 0.75
    if edges['highway'].iloc[ii] == 'secondary_link':
        thicknesses[ii] = 0.75
    if edges['highway'].iloc[ii] == 'trunk':
        thicknesses[ii] = 0.5
    if edges['highway'].iloc[ii] == 'trunk_link':
        thicknesses[ii] = 0.5
    if edges['highway'].iloc[ii] == 'unclassified':
        thicknesses[ii] = 0.1
    if edges['highway'].iloc[ii] == 'tertiary':
        thicknesses[ii] = 0.1
    if edges['highway'].iloc[ii] == 'tertiary_link':
        thicknesses[ii] = 0.1
    if edges['highway'].iloc[ii] == ['unclassified', 'tertiary']:
        thicknesses[ii] = 0.1
    if edges['highway'].iloc[ii] == ['tertiary', 'unclassified']:
        thicknesses[ii] = 0.1

In [ ]:
# PLOT RESULTS #

In [ ]:
vMin=0
vMax=max(max(defra_mod['Total_NO2_23']), max(hourly_means['NO2']))

In [ ]:
# Plot DEFRA modelled NO2
fig, ax = plt.subplots()
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{siunitx}')
sheff.plot(ax=ax, color="white", zorder=0)
lsoas.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=0)
defra_mod.plot(column="Total_NO2_23",
                categorical=False,
                legend=True,
                legend_kwds={"label": r"$\textrm{NO}_2 \textrm{ concentration (\SI{}{\ug}/m} ^3 \textrm{)}$"},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=oranges,
                markersize=60,
                ax=ax,
                zorder=1,
                vmin=vMin)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#plt.savefig(imgOutDir + 'DEFRA_Model_2023.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot DEFRA averaged hourly NO2
fig, ax = plt.subplots()
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{siunitx}')
sheff.plot(ax=ax, color="white", zorder=0)
lsoas.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=0)
hourly_means.plot(column="NO2",
                categorical=False,
                legend=True,
                legend_kwds={"label": r"$\textrm{NO}_2 \textrm{ concentration (\SI{}{\ug}/m} ^3 \textrm{)}$"},
                cmap=oranges,
                markersize=120,
                #linewidth=40,
                ax=ax,
                zorder=1,
                vmin=0)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#plt.savefig(imgOutDir + 'DEFRA_Peak_Feb-Oct_2023.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot DEFRA modelled and hourly averaged NO2
fig, ax = plt.subplots()
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{siunitx}')
sheff.plot(ax=ax, color="white", zorder=0)
lsoas.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=0)
defra_mod.plot(column="Total_NO2_23",
                categorical=False,
                legend=True,
                legend_kwds={"label": r"$\textrm{NO}_2 \textrm{ concentration (\SI{}{\ug}/m} ^3 \textrm{)}$"},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=oranges,
                markersize=60,
                ax=ax,
                zorder=1,
                vmin=vMin,
                vmax=vMax)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
hourly_means.plot(color="black",
                categorical=False,
                legend=False,
                markersize=140,
                ax=ax,
                zorder=1)
hourly_means.plot(column="NO2",
                categorical=False,
                legend=False,
                markersize=120,
                cmap=oranges,
                #linewidth=40,
                ax=ax,
                zorder=2)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#plt.savefig(imgOutDir + 'DEFRA_2023.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot SCC averaged hourly NO2
fig, ax = plt.subplots()
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{siunitx}')
sheff.plot(ax=ax, color="white", zorder=0)
lsoas.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=0)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
scc_means.plot(color="black",
                categorical=False,
                legend=False,
                markersize=80,
                ax=ax,
                zorder=1)
scc_means.plot(column="NO2_ugm3",
                categorical=False,
                legend=True,
                legend_kwds={"label": r"$\textrm{NO}_2 \textrm{ concentration (\SI{}{\ug}/m} ^3 \textrm{)}$"},
                cmap=oranges,
                markersize=60,
                #linewidth=40,
                ax=ax,
                zorder=1,
                vmin=0)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#plt.savefig(imgOutDir + 'SCC_Peak_Feb-Oct_2023.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
vMax = max(max(hourly_means['NO2']), max(scc_means['NO2_ugm3']))

In [ ]:
# Plot DEFRA & SCC averaged hourly NO2
fig, ax = plt.subplots()
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{siunitx}')
sheff.plot(ax=ax, color="#F2F1F1", zorder=0)
lsoas.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=0)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
hourly_means.plot(color="black",
                categorical=False,
                legend=False,
                markersize=m2,
                ax=ax,
                zorder=1)
hourly_means.plot(column="NO2",
                categorical=False,
                legend=False,
                markersize=m1,
                cmap=oranges,
                ax=ax,
                zorder=2)
scc_means.plot(color="black",
                categorical=False,
                legend=False,
                markersize=m2,
                ax=ax,
                zorder=1)
scc_means.plot(column="NO2_ugm3",
                categorical=False,
                legend=True,
                legend_kwds={"label": r"$\textrm{NO}_2 \textrm{ concentration (\SI{}{\ug}/m} ^3 \textrm{)}$"},
                cmap=oranges,
                markersize=m1,
                #linewidth=40,
                ax=ax,
                zorder=1,
                vmin=0,
                vmax=vMax)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)

for x, y, label in zip(scc_means.geometry.x, scc_means.geometry.y, scc_means.label):
    ax.annotate(label, xy=(x, y), xytext=(-1.5, -2.2), textcoords="offset points", size=8)
for x, y, label in zip(hourly_means.geometry.x, hourly_means.geometry.y, hourly_means.label):
    ax.annotate(label, xy=(x, y), xytext=(-1.5, -2.2), textcoords="offset points", size=8)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#plt.savefig(imgOutDir + 'SCC_and_DEFRA_Peak_Feb-Oct_2023.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
vMin = min(min(hourly_means['NO2']), min(scc_means['NO2_ugm3']), min(defra_mod['Total_NO2_23']))
vMax = max(max(hourly_means['NO2']), max(scc_means['NO2_ugm3']), max(defra_mod['Total_NO2_23']))

In [ ]:
# # decrease font size for plots that will be displayed larger
# plt.rcParams.update({'font.size': 10})

In [ ]:
# Plot everything for NO2
fig, ax = plt.subplots()
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{siunitx}')
sheff.plot(ax=ax, color="white", zorder=0)
lsoas.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=0)
defra_mod.plot(column="Total_NO2_23",
                categorical=False,
                legend=True,
                legend_kwds={"label": r"$\textrm{NO}_2 \textrm{ concentration (\SI{}{\ug}/m} ^3 \textrm{)}$"},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=oranges,
                markersize=m1,
                ax=ax,
                zorder=1,
                vmin=0,
                vmax=vMax)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
hourly_means.plot(color="black",
                categorical=False,
                legend=False,
                markersize=m2,
                ax=ax,
                zorder=2)
scc_means.plot(color="black",
                categorical=False,
                legend=False,
                markersize=m2,
                ax=ax,
                zorder=2)
hourly_means.plot(column="NO2",
                categorical=False,
                legend=False,
                markersize=m1,
                cmap=oranges,
                ax=ax,
                zorder=3)
scc_means.plot(column="NO2_ugm3",
                categorical=False,
                legend=False,
                cmap=oranges,
                markersize=m1,
                ax=ax,
                zorder=3,
                vmin=0,
                vmax=vMax)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)

for x, y, label in zip(scc_means.geometry.x, scc_means.geometry.y, scc_means.label):
    ax.annotate(label, xy=(x, y), xytext=(-1.5, -2.2), textcoords="offset points", size=8)
for x, y, label in zip(hourly_means.geometry.x, hourly_means.geometry.y, hourly_means.label):
    ax.annotate(label, xy=(x, y), xytext=(-1.5, -2.2), textcoords="offset points", size=8)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#plt.savefig(imgOutDir + 'SCC_and_DEFRA.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
vMax = 2000

In [ ]:
# Plot flows
fig, ax = plt.subplots()
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{siunitx}')
sheff.plot(ax=ax, color="#F2F1F1", zorder=0)
lsoas.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=0)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
flow.plot(color="black",
                categorical=False,
                legend=False,
                markersize=m2/4,
                ax=ax,
                zorder=2)
flow.plot(column="FlowHour",
                categorical=False,
                legend=True,
                legend_kwds={"label": r"$\textrm{Traffic flow (veh/hour)}$"},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=purples,
                markersize=m1/4,
                ax=ax,
                zorder=3,
                vmin=0,
                vmax=vMax)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#plt.savefig(imgOutDir + 'SCC_Flow_2019.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# WRITE RESULTS #

In [ ]:
#flow.to_csv(dataDir + '')